In [3]:
from langgraph.graph import StateGraph,START,MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages.utils import trim_messages,count_tokens_approximately
from dotenv import load_dotenv

In [4]:
load_dotenv()

True

In [5]:
model=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [6]:
MAX_TOKENS=150

In [ ]:
def call_model(state:MessagesState):
    messages=trim_messages(state["messages"],strategy="last",token_counter=count_tokens_approximately,max_tokens=MAX_TOKENS)
    print("current token count:",count_tokens_approximately(messages=messages))
    for message in messages:
        print(message.content)

    response=model.invoke(messages)
    return {"messages":[response]}


In [ ]:
builder=StateGraph(MessagesState)
builder.add_node("call_model",call_model)
builder.add_edge(START,"call_model")

In [ ]:
checkpointer=InMemorySaver()
graph=builder.compile(checkpointer=checkpointer)

In [ ]:
config={"configurable":{"thread_id":"chat1"}}
result=graph.invoke(
    {"messages":[{"role":"user","content":"Hi my name is Taqadus"}]},
    config
)
result["messages"][-1].content